In [1]:
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent

protein_path = ROOT / "data/processed/human_proteins_with_string.parquet"
string_path = ROOT / "data/raw/string/9606.protein.links.detailed.v12.5.txt.gz"

protein_df = pd.read_parquet(protein_path)
valid_string_ids = set(protein_df["string_id"])

SCORE_THRESHOLD = 700

print("Protein nodes available:", len(protein_df))
print("Chosen STRING score threshold:", SCORE_THRESHOLD)

Protein nodes available: 18841
Chosen STRING score threshold: 700


c:\Users\rajdeep\Desktop\Capstone Project_GNN\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Raw file ko chunks mein read karo; sirf high-confidence usable edges RAM mein rakho
edge_columns = [
    "protein1",
    "protein2",
    "experimental",
    "database",
    "textmining",
    "combined_score",
]

ppi_edge_parts = []

chunk_reader = pd.read_csv(
    string_path,
    sep=r"\s+",
    compression="gzip",
    usecols=edge_columns,
    chunksize=250_000,
)

for chunk in tqdm(chunk_reader, desc="Building high-confidence PPI edges"):
    keep_edge = (
        chunk["protein1"].isin(valid_string_ids)
        & chunk["protein2"].isin(valid_string_ids)
        & (chunk["combined_score"] >= SCORE_THRESHOLD)
    )

    ppi_edge_parts.append(chunk.loc[keep_edge].copy())

ppi_edges = pd.concat(ppi_edge_parts, ignore_index=True)

print("Edges before final validation:", f"{len(ppi_edges):,}")
ppi_edges.head()

Building high-confidence PPI edges: 44it [00:39,  1.13it/s]

Edges before final validation: 533,818


,protein1,protein2,experimental,database,textmining,combined_score
0,9606.ENSP00000000233,9606.ENSP00000158762,46,0,835,839
1,9606.ENSP00000000233,9606.ENSP00000239690,902,0,0,904
2,9606.ENSP00000000233,9606.ENSP00000249923,113,500,609,828
3,9606.ENSP00000000233,9606.ENSP00000256682,840,500,60,995
4,9606.ENSP00000000233,9606.ENSP00000260363,457,500,0,716


In [3]:
# Final graph-quality checks
ppi_edges = ppi_edges[
    ppi_edges["protein1"] != ppi_edges["protein2"]
].copy()

ppi_edges = ppi_edges.drop_duplicates(
    subset=["protein1", "protein2"]
).reset_index(drop=True)

# PyTorch Geometric mein baad mein is weight ko edge attribute ke taur par use kar sakte hain
ppi_edges["edge_weight"] = ppi_edges["combined_score"] / 1000.0

assert ppi_edges["protein1"].isin(valid_string_ids).all()
assert ppi_edges["protein2"].isin(valid_string_ids).all()

graph_string_nodes = set(ppi_edges["protein1"]) | set(ppi_edges["protein2"])

print("Final PPI edges:", f"{len(ppi_edges):,}")
print("Protein nodes connected by an edge:", f"{len(graph_string_nodes):,}")
print("Protein nodes with no retained edge:", f"{len(protein_df) - len(graph_string_nodes):,}")

print("\nCombined-score summary:")
print(ppi_edges["combined_score"].describe())

ppi_edges.head()

Final PPI edges: 533,818
Protein nodes connected by an edge: 16,983
Protein nodes with no retained edge: 1,858

Combined-score summary:
count    533818.000000
mean        865.770757
std          94.259142
min         700.000000
25%         786.000000
50%         871.000000
75%         950.000000
max         999.000000
Name: combined_score, dtype: float64


,protein1,protein2,experimental,database,textmining,combined_score,edge_weight
0,9606.ENSP00000000233,9606.ENSP00000158762,46,0,835,839,0.839
1,9606.ENSP00000000233,9606.ENSP00000239690,902,0,0,904,0.904
2,9606.ENSP00000000233,9606.ENSP00000249923,113,500,609,828,0.828
3,9606.ENSP00000000233,9606.ENSP00000256682,840,500,60,995,0.995
4,9606.ENSP00000000233,9606.ENSP00000260363,457,500,0,716,0.716


In [4]:
output_path = ROOT / "data/processed/ppi_edges_score_700.parquet"

ppi_edges.to_parquet(output_path, index=False)

print("Saved:", output_path)
print(f"File size: {output_path.stat().st_size / 1024**2:.2f} MB")

Saved: c:\Users\rajdeep\Desktop\Capstone Project_GNN\data\processed\ppi_edges_score_700.parquet
File size: 3.90 MB
